# Title
Baseline sklearn and skrub Benchmark
## Purpose
Describe the default benchmark path for median, Ridge, and skrub-backed Ridge models.
## Inputs
`data/gold/listings_gold.xlsx` and the `train_baseline_models` helper.
## Outputs
Benchmark metrics, holdout predictions, and a model card under `results/benchmarks/...` when training is enabled.
## Key Takeaways
This notebook defaults to inspection of existing outputs instead of retraining.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

def resolve_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'config.py').exists():
            return candidate
    raise FileNotFoundError('Could not find config.py in the current path ancestry')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import LISTINGS_GOLD, RESULTS_DIR
from elferspot_listings.modeling.train import train_baseline_models

BENCHMARK_DIR = RESULTS_DIR / 'benchmarks' / '01_baseline_sklearn_skrub'
RUN_TRAINING = False
TRAIN_CATBOOST = False
RUN_CHALLENGERS = False


In [ ]:
gold_exists = LISTINGS_GOLD.exists()
metrics_path = BENCHMARK_DIR / 'metrics.json'
predictions_path = BENCHMARK_DIR / 'predictions.csv'
model_card_path = BENCHMARK_DIR / 'MODEL_CARD.md'

print(f'Gold input: {LISTINGS_GOLD}')
print(f'Gold present: {gold_exists}')
print(f'Benchmark directory: {BENCHMARK_DIR}')

if gold_exists:
    gold_preview = pd.read_excel(LISTINGS_GOLD, nrows=3)
    print('\nGold preview:')
    print(gold_preview.to_string(index=False))
else:
    print('Gold input is missing, so the benchmark stays in dry-run mode.')


In [ ]:
if RUN_TRAINING and gold_exists:
    gold_df = pd.read_excel(LISTINGS_GOLD)
    result = train_baseline_models(
        gold_df,
        BENCHMARK_DIR,
        train_catboost=TRAIN_CATBOOST,
        run_tabpfn=RUN_CHALLENGERS,
        run_autogluon=RUN_CHALLENGERS,
    )
    print(f'Trained models: {sorted(result.metrics)}')
elif metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    predictions = pd.read_csv(predictions_path) if predictions_path.exists() else pd.DataFrame()
    print('Loaded existing benchmark metrics:')
    print(pd.DataFrame(metrics).T.sort_values('mae_eur').to_string())
    if not predictions.empty:
        print('\nPrediction sample:')
        print(predictions.head().to_string(index=False))
    if model_card_path.exists():
        print(f'\nModel card available at: {model_card_path}')
else:
    print('No benchmark outputs found yet. Leave RUN_TRAINING=False for review-only use, or set it to True to regenerate.')
